In [61]:
import pandas as pd
import numpy as np
# import pyodbc
import datetime
import numpy as np
# import matplotlib.pyplot as plt
# import scipy.stats as stats

In [62]:
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
df = pd.read_parquet(".parquet")

In [64]:
# Делаем расчеты показателей по СAMELS 
# Статьи доходов/расходов взяты на накопительной основе за 4 квартала

In [65]:
used_measures = [
    'Активы',
    'Риск. активы и усл. обяз-ва',
    'Коэф.дост-ти СК (k2)',
    'Обяз-ва перед клиентами',
    'Текущие и карт-счета, тыс.тенге',
    'Вклады до востребования юр. и физ. лиц, тыс.тенге',
    'Всего вклады юр. и физ. лиц, тыс.тенге',
    'Активы, приносящие доход',
    'Операционные расходы (на конец периода, тыс. тенге)',
    'Чистый % доход, связанный с получением вознаграждения',
    'Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)',
    'Расходы, не связанные с выплатой вознаграждения (на конец периода, тыс. тенге)',
    'Расходы, связанные с выплатой вознаграждения (на конец периода, тыс. тенге)',
    'Обяз-ва, связ. с выпл.%возн.',
    'Высоколиквидные активы',
    'Итого ссудный портфель',
    'Чистый доход до уплаты подоходного налога (на конец периода, тыс. тенге)',
    'Чистый доход после уплаты подоходного налога (на конец периода, тыс. тенге)',
    'Собственный капитал',
    'Ценные бумаги (на конец периода, тыс. тенге)',
    'Производные финансовые инструменты (на конец периода, тыс. тенге)',
    'Сумма займов, по которым просроченная задолженность свыше 90 дней',
    'Расходы на формирование провизий (на конец периода, тыс. тенге)',
    'Расходы по выплате подоходного налога (на конец периода, тыс. тенге)',
    'Обяз-ва БВУ в ин.валюте',
    'Обязательства',
    'Требования к НБ (на конец периода, тыс. тенге)',
    'Корр. Счета и вклады, размещенные в других БВУ и НБО (на конец периода, тыс. тенге)',
    'Деньги и драг. металлы (на конец периода, тыс. тенге)'
]

In [66]:
df_used = df[df['Measures_normalized'].isin(used_measures)]

In [67]:
df_used = df_used[(df_used.duplicated(subset=['Date', 'BVU', 'Measures_normalized'], keep=False))]

In [68]:
df_used.BVU.unique()

<ArrowStringArray>
[]
Length: 0, dtype: str

In [71]:
def calc_measures(group):
    bvu, quarter = group.name # new
    return pd.DataFrame({
        # 'BVU': [group['BVU'].iloc[0]],
        # 'Quarter': [group['Quarter'].iloc[0]],
        'BVU': [bvu], # new
        'Quarter': [quarter], # new
  
        'Asset_growth': [group.loc[group['Measures_normalized'] == 'Активы', 'Amount_QoQ_growth'].values[0] 
                         if not group.loc[group['Measures_normalized'] == 'Активы', 'Amount_QoQ_growth'].empty else None],
    
        'Average_risk_weight': [(group.loc[group['Measures_normalized'] == 'Риск. активы и усл. обяз-ва', 'Amount'].values[0] /
                                  group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].values[0])
                                 if (not group.loc[group['Measures_normalized'] == 'Риск. активы и усл. обяз-ва', 'Amount'].empty and
                                     not group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].empty) else None],


              
        'Capital buffer (CAR)': [group.loc[group['Measures_normalized'] == 'Коэф.дост-ти СК (k2)', 'Amount'].values[0] 
                         if not group.loc[group['Measures_normalized'] == 'Коэф.дост-ти СК (k2)', 'Amount'].empty else None],

        
    'Deposit_growth': [group.loc[group['Measures_normalized'] == 'Обяз-ва перед клиентами', 'Amount_QoQ_growth'].values[0] 
                         if not group.loc[group['Measures_normalized'] == 'Обяз-ва перед клиентами', 'Amount_QoQ_growth'].empty else None],

        'Demand Deposit Ratio': [(
    group.loc[group['Measures_normalized'].isin(['Текущие и карт-счета, тыс.тенге', 'Вклады до востребования юр. и физ. лиц, тыс.тенге']), 'Amount'].sum() /
    group.loc[group['Measures_normalized'] == 'Всего вклады юр. и физ. лиц, тыс.тенге', 'Amount'].sum()
) if (not group.loc[group['Measures_normalized'].isin(['Текущие и карт-счета, тыс.тенге', 'Вклады до востребования юр. и физ. лиц, тыс.тенге']), 'Amount'].empty and
      not group.loc[group['Measures_normalized'] == 'Всего вклады юр. и физ. лиц, тыс.тенге', 'Amount'].empty) else None], 
   
        
            'Earning assets to total': [(group.loc[group['Measures_normalized'] == 'Активы, приносящие доход', 'Amount'].values[0] /
                                  group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].values[0])
                                 if (not group.loc[group['Measures_normalized'] == 'Активы, приносящие доход', 'Amount'].empty and
                                     not group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].empty) else None],


'Efficiency ratio (version 1)': [(
    group.loc[group['Measures_normalized'] == 'Операционные расходы (на конец периода, тыс. тенге)', 'Value_4Q_sum'].sum() /
    group.loc[group['Measures_normalized'].isin([
        'Чистый % доход, связанный с получением вознаграждения',
        'Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)'
    ]), 'Value_4Q_sum'].sum()
) if (not group.loc[group['Measures_normalized'] == 'Операционные расходы (на конец периода, тыс. тенге)', 'Value_4Q_sum'].empty and
      not group.loc[group['Measures_normalized'].isin([
          'Чистый % доход, связанный с получением вознаграждения',
          'Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)'
      ]), 'Value_4Q_sum'].empty) else None],



        

'Efficiency ratio (version 2)': [(
    group.loc[group['Measures_normalized'] == 'Расходы, не связанные с выплатой вознаграждения (на конец периода, тыс. тенге)', 'Value_4Q_sum'].sum() /
    group.loc[group['Measures_normalized'].isin([
        'Чистый % доход, связанный с получением вознаграждения',
        'Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)'
    ]), 'Value_4Q_sum'].sum()
) if (not group.loc[group['Measures_normalized'] == 'Расходы, не связанные с выплатой вознаграждения (на конец периода, тыс. тенге)', 'Value_4Q_sum'].empty and
      not group.loc[group['Measures_normalized'].isin([
          'Чистый % доход, связанный с получением вознаграждения',
          'Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)'
      ]), 'Value_4Q_sum'].empty) else None],

        

        
  'Cost of funds': [(group.loc[group['Measures_normalized'] == 'Расходы, связанные с выплатой вознаграждения (на конец периода, тыс. тенге)', 'Value_4Q_sum'].values[0] /
                                  group.loc[group['Measures_normalized'] == 'Обяз-ва, связ. с выпл.%возн.', 'Amount_5Q_avg'].values[0])
                                 if (not group.loc[group['Measures_normalized'] == 'Расходы, связанные с выплатой вознаграждения (на конец периода, тыс. тенге)', 'Value_4Q_sum'].empty and
                                     not group.loc[group['Measures_normalized'] == 'Обяз-ва, связ. с выпл.%возн.', 'Amount_5Q_avg'].empty) else None],
        


        
    'Liquidity ratio': [(group.loc[group['Measures_normalized'] == 'Высоколиквидные активы', 'Amount'].values[0] /
                                  group.loc[group['Measures_normalized'] == 'Риск. активы и усл. обяз-ва', 'Amount'].values[0])
                                 if (not group.loc[group['Measures_normalized'] == 'Высоколиквидные активы', 'Amount'].empty and
                                     not group.loc[group['Measures_normalized'] == 'Риск. активы и усл. обяз-ва', 'Amount'].empty) else None],

        #Loan 
        
    'Loan_growth': [group.loc[group['Measures_normalized'] == 'Итого ссудный портфель', 'Amount_QoQ_growth'].values[0] 
                         if not group.loc[group['Measures_normalized'] == 'Итого ссудный портфель', 'Amount_QoQ_growth'].empty else None],

  'Loans to deposits': [(group.loc[group['Measures_normalized'] == 'Итого ссудный портфель', 'Amount'].values[0] /
                                  group.loc[group['Measures_normalized'] == 'Обяз-ва перед клиентами', 'Amount'].values[0])
                                 if (not group.loc[group['Measures_normalized'] == 'Итого ссудный портфель', 'Amount'].empty and
                                     not group.loc[group['Measures_normalized'] == 'Обяз-ва перед клиентами', 'Amount'].empty) else None],



    
    'Net interest margin': [(group.loc[group['Measures_normalized'] == 'Чистый % доход, связанный с получением вознаграждения', 'Value_4Q_sum'].values[0] /
                                  group.loc[group['Measures_normalized'] == 'Активы, приносящие доход', 'Amount_5Q_avg'].values[0])
                                 if (not group.loc[group['Measures_normalized'] == 'Чистый % доход, связанный с получением вознаграждения', 'Value_4Q_sum'].empty and
                                     not group.loc[group['Measures_normalized'] == 'Активы, приносящие доход', 'Amount_5Q_avg'].empty) else None],
               
        

   'Non-interest income': [group.loc[group['Measures_normalized'] =='Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)', 'Amount_QoQ_growth'].values[0] 
                         if not group.loc[group['Measures_normalized'] == 'Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)', 'Amount_QoQ_growth'].empty else None],

    'Pre-tax net income': [group.loc[group['Measures_normalized'] =='Чистый доход до уплаты подоходного налога (на конец периода, тыс. тенге)', 'Amount_QoQ_growth'].values[0] 
                         if not group.loc[group['Measures_normalized'] == 'Чистый доход до уплаты подоходного налога (на конец периода, тыс. тенге)', 'Amount_QoQ_growth'].empty else None],

    
        'ROE': [(group.loc[group['Measures_normalized'] == 'Чистый доход после уплаты подоходного налога (на конец периода, тыс. тенге)', 'Value_4Q_sum'].values[0] /
                                  group.loc[group['Measures_normalized'] == 'Собственный капитал', 'Amount_5Q_avg'].values[0])
                                 if (not group.loc[group['Measures_normalized'] == 'Чистый доход после уплаты подоходного налога (на конец периода, тыс. тенге)', 'Value_4Q_sum'].empty and
                                     not group.loc[group['Measures_normalized'] == 'Собственный капитал', 'Amount_5Q_avg'].empty) else None],   

     'ROA': [(group.loc[group['Measures_normalized'] == 'Чистый доход после уплаты подоходного налога (на конец периода, тыс. тенге)', 'Value_4Q_sum'].values[0] /
                                  group.loc[group['Measures_normalized'] == 'Активы', 'Amount_5Q_avg'].values[0])
                                 if (not group.loc[group['Measures_normalized'] == 'Чистый доход после уплаты подоходного налога (на конец периода, тыс. тенге)', 'Value_4Q_sum'].empty and
                                     not group.loc[group['Measures_normalized'] == 'Активы', 'Amount_5Q_avg'].empty) else None],  



          'Trading book assets': [(
    group.loc[group['Measures_normalized'].isin(['Ценные бумаги (на конец периода, тыс. тенге)', 'Производные финансовые инструменты (на конец периода, тыс. тенге)']), 'Amount'].sum() /
    group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].sum()
) if (not group.loc[group['Measures_normalized'].isin(['Ценные бумаги (на конец периода, тыс. тенге)', 'Производные финансовые инструменты (на конец периода, тыс. тенге)']), 'Amount'].empty and
      not group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].empty) else None], 



    'Equity/Total assets': [(group.loc[group['Measures_normalized'] == 'Собственный капитал', 'Amount'].values[0] /
                                  group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].values[0])
                                 if (not group.loc[group['Measures_normalized'] == 'Собственный капитал', 'Amount'].empty and
                                     not group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].empty) else None],


        

         'NPL/Total Loans': [(group.loc[group['Measures_normalized'] == 'Сумма займов, по которым просроченная задолженность свыше 90 дней', 'Amount'].values[0] /
                                  group.loc[group['Measures_normalized'] == 'Итого ссудный портфель', 'Amount'].values[0])
                                 if (not group.loc[group['Measures_normalized'] == 'Сумма займов, по которым просроченная задолженность свыше 90 дней', 'Amount'].empty and
                                     not group.loc[group['Measures_normalized'] == 'Итого ссудный портфель', 'Amount'].empty) else None],   
        

'Provision Expense Ratio': [(
    group.loc[group['Measures_normalized'] == 'Расходы на формирование провизий (на конец периода, тыс. тенге)', 'Value_4Q_sum'].sum() /
    group.loc[group['Measures_normalized'].isin([
        'Расходы, связанные с выплатой вознаграждения (на конец периода, тыс. тенге)',
        'Расходы, не связанные с выплатой вознаграждения (на конец периода, тыс. тенге)',
        'Расходы по выплате подоходного налога (на конец периода, тыс. тенге)'
    ]), 'Value_4Q_sum'].sum()
) if (not group.loc[group['Measures_normalized'] == 'Расходы на формирование провизий (на конец периода, тыс. тенге)', 'Value_4Q_sum'].empty and
      not group.loc[group['Measures_normalized'].isin([
          'Расходы, связанные с выплатой вознаграждения (на конец периода, тыс. тенге)',
          'Расходы, не связанные с выплатой вознаграждения (на конец периода, тыс. тенге)',
          'Расходы по выплате подоходного налога (на конец периода, тыс. тенге)'
      ]), 'Value_4Q_sum'].empty) else None],



         'FX Liabilities Share': [(group.loc[group['Measures_normalized'] == 'Обяз-ва БВУ в ин.валюте', 'Amount'].values[0] /
                                  group.loc[group['Measures_normalized'] == 'Обязательства', 'Amount'].values[0])
                                 if (not group.loc[group['Measures_normalized'] == 'Обяз-ва БВУ в ин.валюте', 'Amount'].empty and
                                     not group.loc[group['Measures_normalized'] == 'Обязательства', 'Amount'].empty) else None],           


            'Cash to Assets': [(
    group.loc[group['Measures_normalized'].isin(['Требования к НБ (на конец периода, тыс. тенге)', 
                                                 'Корр. Счета и вклады, размещенные в других БВУ и НБО (на конец периода, тыс. тенге)',
                                                 'Деньги и драг. металлы (на конец периода, тыс. тенге)'
                                                ]), 'Amount'].sum() /
    group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].sum()
) if (not group.loc[group['Measures_normalized'].isin(['Требования к НБ (на конец периода, тыс. тенге)', 
                                                       'Корр. Счета и вклады, размещенные в других БВУ и НБО (на конец периода, тыс. тенге)',
                                                       'Деньги и драг. металлы (на конец периода, тыс. тенге)'
                                                      ]), 'Amount'].empty and
      not group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].empty) else None]


        
    
    })

# Apply per BVU + Quarter
df_result = df.groupby(['BVU', 'Quarter'], group_keys=False).apply(calc_measures)

In [72]:
# Делаем еще копию расчетов показателей по СAMELS 
# Статьи доходов/расходов взяты на квартальной основе (на точку)

In [73]:
# def calc_measures(group):
#     return pd.DataFrame({
#         'BVU': [group['BVU'].iloc[0]],
#         'Quarter': [group['Quarter'].iloc[0]],
  
#         'Asset_growth': [group.loc[group['Measures_normalized'] == 'Активы', 'Amount_QoQ_growth'].values[0] 
#                          if not group.loc[group['Measures_normalized'] == 'Активы', 'Amount_QoQ_growth'].empty else None],
    
#         'Average_risk_weight': [(group.loc[group['Measures_normalized'] == 'Риск. активы и усл. обяз-ва', 'Amount'].values[0] /
#                                   group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].values[0])
#                                  if (not group.loc[group['Measures_normalized'] == 'Риск. активы и усл. обяз-ва', 'Amount'].empty and
#                                      not group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].empty) else None],


              
#         'Capital buffer (CAR)': [group.loc[group['Measures_normalized'] == 'Коэф.дост-ти СК (k2)', 'Amount'].values[0] 
#                          if not group.loc[group['Measures_normalized'] == 'Коэф.дост-ти СК (k2)', 'Amount'].empty else None],

        
#     'Deposit_growth': [group.loc[group['Measures_normalized'] == 'Обяз-ва перед клиентами', 'Amount_QoQ_growth'].values[0] 
#                          if not group.loc[group['Measures_normalized'] == 'Обяз-ва перед клиентами', 'Amount_QoQ_growth'].empty else None],

#         'Demand Deposit Ratio': [(
#     group.loc[group['Measures_normalized'].isin(['Текущие и карт-счета, тыс.тенге', 'Вклады до востребования юр. и физ. лиц, тыс.тенге']), 'Amount'].sum() /
#     group.loc[group['Measures_normalized'] == 'Всего вклады юр. и физ. лиц, тыс.тенге', 'Amount'].sum()
# ) if (not group.loc[group['Measures_normalized'].isin(['Текущие и карт-счета, тыс.тенге', 'Вклады до востребования юр. и физ. лиц, тыс.тенге']), 'Amount'].empty and
#       not group.loc[group['Measures_normalized'] == 'Всего вклады юр. и физ. лиц, тыс.тенге', 'Amount'].empty) else None], 
   
        
#             'Earning assets to total': [(group.loc[group['Measures_normalized'] == 'Активы, приносящие доход', 'Amount'].values[0] /
#                                   group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].values[0])
#                                  if (not group.loc[group['Measures_normalized'] == 'Активы, приносящие доход', 'Amount'].empty and
#                                      not group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].empty) else None],


# 'Efficiency ratio (version 1)': [(
#     group.loc[group['Measures_normalized'] == 'Операционные расходы (на конец периода, тыс. тенге)', 'Amount'].sum() /
#     group.loc[group['Measures_normalized'].isin([
#         'Чистый % доход, связанный с получением вознаграждения',
#         'Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)'
#     ]), 'Amount'].sum()
# ) if (not group.loc[group['Measures_normalized'] == 'Операционные расходы (на конец периода, тыс. тенге)', 'Amount'].empty and
#       not group.loc[group['Measures_normalized'].isin([
#           'Чистый % доход, связанный с получением вознаграждения',
#           'Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)'
#       ]), 'Amount'].empty) else None],



        

# 'Efficiency ratio (version 2)': [(
#     group.loc[group['Measures_normalized'] == 'Расходы, не связанные с выплатой вознаграждения (на конец периода, тыс. тенге)', 'Amount'].sum() /
#     group.loc[group['Measures_normalized'].isin([
#         'Чистый % доход, связанный с получением вознаграждения',
#         'Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)'
#     ]), 'Amount'].sum()
# ) if (not group.loc[group['Measures_normalized'] == 'Расходы, не связанные с выплатой вознаграждения (на конец периода, тыс. тенге)', 'Amount'].empty and
#       not group.loc[group['Measures_normalized'].isin([
#           'Чистый % доход, связанный с получением вознаграждения',
#           'Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)'
#       ]), 'Amount'].empty) else None],

        

        
#   'Cost of funds': [(group.loc[group['Measures_normalized'] == 'Расходы, связанные с выплатой вознаграждения (на конец периода, тыс. тенге)', 'Value_4Q_sum'].values[0] /
#                                   group.loc[group['Measures_normalized'] == 'Обяз-ва, связ. с выпл.%возн.', 'Amount_5Q_avg'].values[0])
#                                  if (not group.loc[group['Measures_normalized'] == 'Расходы, связанные с выплатой вознаграждения (на конец периода, тыс. тенге)', 'Value_4Q_sum'].empty and
#                                      not group.loc[group['Measures_normalized'] == 'Обяз-ва, связ. с выпл.%возн.', 'Amount_5Q_avg'].empty) else None],
        


        
#     'Liquidity ratio': [(group.loc[group['Measures_normalized'] == 'Высоколиквидные активы', 'Amount'].values[0] /
#                                   group.loc[group['Measures_normalized'] == 'Риск. активы и усл. обяз-ва', 'Amount'].values[0])
#                                  if (not group.loc[group['Measures_normalized'] == 'Высоколиквидные активы', 'Amount'].empty and
#                                      not group.loc[group['Measures_normalized'] == 'Риск. активы и усл. обяз-ва', 'Amount'].empty) else None],

#         #Loan 
        
#     'Loan_growth': [group.loc[group['Measures_normalized'] == 'Итого ссудный портфель', 'Amount_QoQ_growth'].values[0] 
#                          if not group.loc[group['Measures_normalized'] == 'Итого ссудный портфель', 'Amount_QoQ_growth'].empty else None],

#   'Loans to deposits': [(group.loc[group['Measures_normalized'] == 'Итого ссудный портфель', 'Amount'].values[0] /
#                                   group.loc[group['Measures_normalized'] == 'Обяз-ва перед клиентами', 'Amount'].values[0])
#                                  if (not group.loc[group['Measures_normalized'] == 'Итого ссудный портфель', 'Amount'].empty and
#                                      not group.loc[group['Measures_normalized'] == 'Обяз-ва перед клиентами', 'Amount'].empty) else None],



    
#     'Net interest margin': [(group.loc[group['Measures_normalized'] == 'Чистый % доход, связанный с получением вознаграждения', 'Value_4Q_sum'].values[0] /
#                                   group.loc[group['Measures_normalized'] == 'Активы, приносящие доход', 'Amount_5Q_avg'].values[0])
#                                  if (not group.loc[group['Measures_normalized'] == 'Чистый % доход, связанный с получением вознаграждения', 'Value_4Q_sum'].empty and
#                                      not group.loc[group['Measures_normalized'] == 'Активы, приносящие доход', 'Amount_5Q_avg'].empty) else None],
               
        

#    'Non-interest income': [group.loc[group['Measures_normalized'] =='Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)', 'Amount_QoQ_growth'].values[0] 
#                          if not group.loc[group['Measures_normalized'] == 'Доходы, не связанные с получением вознаграждения (на конец периода, тыс. тенге)', 'Amount_QoQ_growth'].empty else None],

#     'Pre-tax net income': [group.loc[group['Measures_normalized'] =='Чистый доход до уплаты подоходного налога (на конец периода, тыс. тенге)', 'Amount_QoQ_growth'].values[0] 
#                          if not group.loc[group['Measures_normalized'] == 'Чистый доход до уплаты подоходного налога (на конец периода, тыс. тенге)', 'Amount_QoQ_growth'].empty else None],

    
#         'ROE': [(group.loc[group['Measures_normalized'] == 'Чистый доход после уплаты подоходного налога (на конец периода, тыс. тенге)', 'Value_4Q_sum'].values[0] /
#                                   group.loc[group['Measures_normalized'] == 'Собственный капитал', 'Amount_5Q_avg'].values[0])
#                                  if (not group.loc[group['Measures_normalized'] == 'Чистый доход после уплаты подоходного налога (на конец периода, тыс. тенге)', 'Value_4Q_sum'].empty and
#                                      not group.loc[group['Measures_normalized'] == 'Собственный капитал', 'Amount_5Q_avg'].empty) else None],   

#      'ROA': [(group.loc[group['Measures_normalized'] == 'Чистый доход после уплаты подоходного налога (на конец периода, тыс. тенге)', 'Value_4Q_sum'].values[0] /
#                                   group.loc[group['Measures_normalized'] == 'Активы', 'Amount_5Q_avg'].values[0])
#                                  if (not group.loc[group['Measures_normalized'] == 'Чистый доход после уплаты подоходного налога (на конец периода, тыс. тенге)', 'Value_4Q_sum'].empty and
#                                      not group.loc[group['Measures_normalized'] == 'Активы', 'Amount_5Q_avg'].empty) else None],  



#           'Trading book assets': [(
#     group.loc[group['Measures_normalized'].isin(['Ценные бумаги (на конец периода, тыс. тенге)', 'Производные финансовые инструменты (на конец периода, тыс. тенге)']), 'Amount'].sum() /
#     group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].sum()
# ) if (not group.loc[group['Measures_normalized'].isin(['Ценные бумаги (на конец периода, тыс. тенге)', 'Производные финансовые инструменты (на конец периода, тыс. тенге)']), 'Amount'].empty and
#       not group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].empty) else None], 



#     'Equity/Total assets': [(group.loc[group['Measures_normalized'] == 'Собственный капитал', 'Amount'].values[0] /
#                                   group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].values[0])
#                                  if (not group.loc[group['Measures_normalized'] == 'Собственный капитал', 'Amount'].empty and
#                                      not group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].empty) else None],


        

#          'NPL/Total Loans': [(group.loc[group['Measures_normalized'] == 'Сумма займов, по которым просроченная задолженность свыше 90 дней', 'Amount'].values[0] /
#                                   group.loc[group['Measures_normalized'] == 'Итого ссудный портфель', 'Amount'].values[0])
#                                  if (not group.loc[group['Measures_normalized'] == 'Сумма займов, по которым просроченная задолженность свыше 90 дней', 'Amount'].empty and
#                                      not group.loc[group['Measures_normalized'] == 'Итого ссудный портфель', 'Amount'].empty) else None],   
        

# 'Provision Expense Ratio': [(
#     group.loc[group['Measures_normalized'] == 'Расходы на формирование провизий (на конец периода, тыс. тенге)', 'Amount'].sum() /
#     group.loc[group['Measures_normalized'].isin([
#         'Расходы, связанные с выплатой вознаграждения (на конец периода, тыс. тенге)',
#         'Расходы, не связанные с выплатой вознаграждения (на конец периода, тыс. тенге)',
#         'Расходы по выплате подоходного налога (на конец периода, тыс. тенге)'
#     ]), 'Amount'].sum()
# ) if (not group.loc[group['Measures_normalized'] == 'Расходы на формирование провизий (на конец периода, тыс. тенге)', 'Amount'].empty and
#       not group.loc[group['Measures_normalized'].isin([
#           'Расходы, связанные с выплатой вознаграждения (на конец периода, тыс. тенге)',
#           'Расходы, не связанные с выплатой вознаграждения (на конец периода, тыс. тенге)',
#           'Расходы по выплате подоходного налога (на конец периода, тыс. тенге)'
#       ]), 'Amount'].empty) else None],



#          'FX Liabilities Share': [(group.loc[group['Measures_normalized'] == 'Обяз-ва БВУ в ин.валюте', 'Amount'].values[0] /
#                                   group.loc[group['Measures_normalized'] == 'Обязательства', 'Amount'].values[0])
#                                  if (not group.loc[group['Measures_normalized'] == 'Обяз-ва БВУ в ин.валюте', 'Amount'].empty and
#                                      not group.loc[group['Measures_normalized'] == 'Обязательства', 'Amount'].empty) else None],           


#             'Cash to Assets': [(
#     group.loc[group['Measures_normalized'].isin(['Требования к НБ (на конец периода, тыс. тенге)', 
#                                                  'Корр. Счета и вклады, размещенные в других БВУ и НБО (на конец периода, тыс. тенге)',
#                                                  'Деньги и драг. металлы (на конец периода, тыс. тенге)'
#                                                 ]), 'Amount'].sum() /
#     group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].sum()
# ) if (not group.loc[group['Measures_normalized'].isin(['Требования к НБ (на конец периода, тыс. тенге)', 
#                                                        'Корр. Счета и вклады, размещенные в других БВУ и НБО (на конец периода, тыс. тенге)',
#                                                        'Деньги и драг. металлы (на конец периода, тыс. тенге)'
#                                                       ]), 'Amount'].empty and
#       not group.loc[group['Measures_normalized'] == 'Активы', 'Amount'].empty) else None]


        
    
#     })

# # Apply per BVU + Quarter
# df_result_QUARTER = df.groupby(['BVU', 'Quarter'], group_keys=False).apply(calc_measures)

In [74]:
# OUTPUT RESULTS ----------------------------------------------------

In [ ]:
df_result.to_excel('.xlsx')